# Parameterized Queries

As discussed in class, many Visual Analytics systems function as wrappers around database queries, including those designed for graph data. In this part of the assignment, you will implement a parameterized query, allowing users to customize query inputs dynamically. This will help you understand how parameterization work, which is the start of implementing your own system in the next part of the assignment.

In [1]:
%matplotlib inline
import os
import sys
import networkx as nx
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from utils import setup_database

In [2]:
connection = setup_database('../tmp', delete_existing=False)

Loading graph database
Removing lock file
Removed stale lock file: ../tmp/.lock


## Building Parameterized Graph Query
1. Set up the output area (already done for you)
2. Make a function that constructs a parameterized query
3. Use that function to query the database
4. Visualize the query results
5. Build interaction elements to configure parameterized query

In [3]:
# TODO: construct a Cypher query with a parameter to filter nodes or relationships in the graph.
def construct_query(parameter_value, operator):
    return f"""
    MATCH (m:Movie)
    WHERE m.year {operator} {parameter_value}
    MATCH (m)-[r]-(n)
    RETURN m, r, n
    LIMIT 100
    """

In [4]:
def visualize_results(parameter_value, operator):
    plt.close('all')
    
    query = construct_query(parameter_value, operator)
    results = connection.execute(query)
    
    graph = results.get_as_networkx(directed=False)

    if graph.number_of_nodes() == 0:
        print(f"No results found for parameter value: {parameter_value}")
        return
    
    print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges")

    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(graph, seed=42, iterations=20)

    nx.draw(graph, pos, 
            ax=ax,
            with_labels=False, 
            node_size=30,
            edge_color='gray',
            alpha=0.6)

    ax.set_title(f"Movie Network (Movies with year {operator} {parameter_value})")
    plt.tight_layout()
    plt.show()

In [ ]:
interactive_plot = widgets.interactive(
    visualize_results,
    parameter_value=widgets.IntSlider(value=2000, min=1950, max=2024, step=1, description='Year:'),
    operator=widgets.Dropdown(options=['>=', '<=', '=', '>', '<'], value='>=', description='Operator:')
)

display(interactive_plot)

interactive(children=(IntSlider(value=2000, description='Year:', max=2024, min=1950), Dropdown(description='Op…